# In Class Activity April 14th, 2026

In [ ]:
# pip install optuna

  Using cached colorlog-6.10.1-py3-none-any.whl.metadata (11 kB)
Using cached colorlog-6.10.1-py3-none-any.whl (11 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [optuna]
Note: you may need to restart the kernel to use updated packages.


### Importing libraries, preparing data, initial EDA

In [2]:
# importing libraries (feel free to add more if you want to explore other things)
import numpy as np
import pandas as pd
import sweetviz as sv
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
from xgboost import XGBClassifier
from sklearn.metrics import f1_score, accuracy_score, classification_report
import optuna


In [8]:
# importing data
adult = pd.read_csv('data/adult.csv')
adult.head(20)

,age,workclass,fnlwgt,education,educational-num,marital-status,occupation,relationship,race,gender,capital-gain,capital-loss,hours-per-week,native-country,income
0,25,Private,226802,11th,7,Never-married,Machine-op-inspct,Own-child,Black,Male,0,0,40,United-States,<=50K
1,38,Private,89814,HS-grad,9,Married-civ-spouse,Farming-fishing,Husband,White,Male,0,0,50,United-States,<=50K
2,28,Local-gov,336951,Assoc-acdm,12,Married-civ-spouse,Protective-serv,Husband,White,Male,0,0,40,United-States,>50K
3,44,Private,160323,Some-college,10,Married-civ-spouse,Machine-op-inspct,Husband,Black,Male,7688,0,40,United-States,>50K
4,18,?,103497,Some-college,10,Never-married,?,Own-child,White,Female,0,0,30,United-States,<=50K
5,34,Private,198693,10th,6,Never-married,Other-service,Not-in-family,White,Male,0,0,30,United-States,<=50K
6,29,?,227026,HS-grad,9,Never-married,?,Unmarried,Black,Male,0,0,40,United-States,<=50K
7,63,Self-emp-not-inc,104626,Prof-school,15,Married-civ-spouse,Prof-specialty,Husband,White,Male,3103,0,32,United-States,>50K
8,24,Private,369667,Some-college,10,Never-married,Other-service,Unmarried,White,Female,0,0,40,United-States,<=50K
9,55,Private,104996,7th-8th,4,Married-civ-spouse,Craft-repair,Husband,White,Male,0,0,10,United-States,<=50K


In [ ]:
# initial EDA with sweetviz
report = sv.analyze(adult)
report.show_html('sweet_report.html')

# you are welcome to replace this cell with your own EDA, but make sure to include
# some visualizations and insights about the data


                                             |          | [  0%]   00:00 -> (? left)

Report sweet_report.html was generated! NOTEBOOK/COLAB USERS: the web browser MAY not pop up, regardless, the report IS saved in your notebook/colab files.


### In the markdown cell below describe what you learned from your EDA and how it will inform your modeling decisions





In the preliminary analysis of the income by demographic dataset, one significant finding was that a lot of the features have a skewed distribution, pointing to multiple areas where there could be class imbalance. There is a greater representation of 20-40 year olds with the average person being ~38 years old, with age having a .58 pearson correlation with marriage status. Since there is a mix of categorical and numerical values, we will need to watch for high cardinality bias, as well as employ tree based models.

### Data Preprocessing (minimal) and Baseline Model

In [9]:
# data cleaning and preprocessing

# changing ? to NaN
adult = adult.replace('?', np.nan)

#education and education num are redundant, so we can drop one of them
adult = adult.drop(columns='education', errors='ignore')


# target variable is income with 2 levels, so we can encode it as 0 and 1
adult['income'] = adult['income'].apply(lambda x: 1 if x == '>50K' else 0)

# change dtype categorical variables to category
categorical_cols = adult.select_dtypes(include='object').columns
adult[categorical_cols] = adult[categorical_cols].astype('category')


adult.head(20)

,age,workclass,fnlwgt,educational-num,marital-status,occupation,relationship,race,gender,capital-gain,capital-loss,hours-per-week,native-country,income
0,25,Private,226802,7,Never-married,Machine-op-inspct,Own-child,Black,Male,0,0,40,United-States,0
1,38,Private,89814,9,Married-civ-spouse,Farming-fishing,Husband,White,Male,0,0,50,United-States,0
2,28,Local-gov,336951,12,Married-civ-spouse,Protective-serv,Husband,White,Male,0,0,40,United-States,1
3,44,Private,160323,10,Married-civ-spouse,Machine-op-inspct,Husband,Black,Male,7688,0,40,United-States,1
4,18,NaN,103497,10,Never-married,NaN,Own-child,White,Female,0,0,30,United-States,0
5,34,Private,198693,6,Never-married,Other-service,Not-in-family,White,Male,0,0,30,United-States,0
6,29,NaN,227026,9,Never-married,NaN,Unmarried,Black,Male,0,0,40,United-States,0
7,63,Self-emp-not-inc,104626,15,Married-civ-spouse,Prof-specialty,Husband,White,Male,3103,0,32,United-States,1
8,24,Private,369667,10,Never-married,Other-service,Unmarried,White,Female,0,0,40,United-States,0
9,55,Private,104996,4,Married-civ-spouse,Craft-repair,Husband,White,Male,0,0,10,United-States,0


In [10]:
# defining features and target variable
X = adult.drop('income', axis=1)
y = adult['income']

# splitting data into train and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=True, 
                                                    random_state=42, stratify=y)

In [33]:
# building xgboost default model and evaluating with stratified k-fold cross validation
xgb_cv = XGBClassifier(enable_categorical=True, random_state=42)
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(xgb_cv, X, y, cv=skf, scoring='f1')
print(f'Cross-validated f1 scores: {cv_scores}')
print(f'Mean f1 score: {cv_scores.mean()}') 


Cross-validated f1 scores: [0.70680507 0.70892566 0.70898981 0.72424942 0.71086556]
Mean f1 score: 0.7119671046056588


### Use the markdown cell below to describe your baseline model performance and how you will try to improve it

The model's baseline performance shows an average f1 score of ~0.71. A few steps to remedy this performance is to handle the class imbalance by using scale_pos_weight, an XGBoost hyperparameter knob. An additional hyperparameter knob would be using max_depth, which helps to control tree complexity and helps to avoid overfitting on the demographic splits. Finally, we will use learning rate, stabilizing training and improves generalizability.

### Model feature exploration

In the code cell below, explore different features of XGBoost and how they work (e.g. scale_pos_weight, max_depth, learning_rate).
Use stratified k-fold cross or repeated stratified k-fold cross validation with your model building. 
You should explore at least 3 different features of XGBoost.
Identify the model that performs best.

In [36]:
# your code here

from sklearn.metrics import accuracy_score, recall_score, f1_score

param_grid = {
  'scale_pos_weight': [1, 2],
  'max_depth': [3, 5],
  'learning_rate': [0.01, 0.1]
}
gscv = GridSearchCV(xgb_cv, param_grid=param_grid)
gscv.fit(X_train, y_train)
gscv.best_params_

y_pred = gscv.best_estimator_.predict(X_test)

print('best param combo:', gscv.best_params_)
print('Test Accuracy:', accuracy_score(y_test, y_pred))
print('Test Recall:', recall_score(y_test, y_pred))
print('Test F1:', f1_score(y_test, y_pred))

best param combo: {'learning_rate': 0.1, 'max_depth': 5, 'scale_pos_weight': 1}
Test Accuracy: 0.8754222540689938
Test Recall: 0.6449957228400343
Test F1: 0.7124970470115757


### Tuning with GridSearchCV

Use the code cell below to set up your parameter grid and run GridSearchCV with your preferred model from above. You should tune 4-5 hyperparameters utilizing cross validation. Train a final model using the best hyperparameters and report your model performance.

In [48]:
# your code here

from sklearn.metrics import accuracy_score, recall_score, f1_score

param_grid = {
  'scale_pos_weight': [1, 2, 3],
  'max_depth': [3, 5, 7],
  'learning_rate': [0.01, 0.1, 0.2],
  'gamma': [0.5, .7, .9]
}
gscv = GridSearchCV(xgb_cv, param_grid=param_grid)
gscv.fit(X_train, y_train)
gscv.best_params_

y_pred = gscv.best_estimator_.predict(X_test)

print('best param combo:', gscv.best_params_)
print('Test Accuracy:', accuracy_score(y_test, y_pred))
print('Test Recall:', recall_score(y_test, y_pred))
print('Test F1:', f1_score(y_test, y_pred))

best param combo: {'gamma': 0.7, 'learning_rate': 0.2, 'max_depth': 5, 'scale_pos_weight': 1}
Test Accuracy: 0.8748080663322756
Test Recall: 0.6522668947818648
Test F1: 0.7137842265387315


### Tuning with RandomizedSearchCV

Using the code cell below as a starting point, tune your preferred model from above. Tune the same 4-5 hyperparameters from above utilizing cross validation. Train a final model using the best hyperparameters and report your model performance.

In [49]:
# tuning xgboost classifier with RandomizedSearchCV (tune more parameters than shown here)
param_dist = {
    'scale_pos_weight': np.linspace(1.0, 10.0, num=10),
    'max_depth': np.arange(3, 11),
    'learning_rate': np.linspace(0.01, 0.3, num=10),
    'gamma': np.linspace(.5, 1, num=4),
    'min_child_weight': np.linspace(1, 5, num=4)
}

# replace this placeholder model with your preferred model from above

xgb_random = RandomizedSearchCV(XGBClassifier(random_state=42, enable_categorical=True),
                                param_distributions=param_dist, n_iter=20, cv=skf, scoring='f1', random_state=42)
xgb_random.fit(X_train, y_train)
print(f'Best parameters from RandomizedSearchCV: {xgb_random.best_params_}')
print(f'Best F1 score from RandomizedSearchCV: {xgb_random.best_score_}')   

# build your preferred model from above using best parameters from your RandomizedSearchCV
# and evaluate on the test set

xgb_random_best = xgb_random.best_estimator_
xgb_random_best.fit(X_train, y_train)
y_pred_random = xgb_random_best.predict(X_test)
print(f'Classification report for RandomizedSearchCV-tuned model:\n{classification_report(y_test, y_pred_random)}')


Best parameters from RandomizedSearchCV: {'scale_pos_weight': np.float64(2.0), 'min_child_weight': np.float64(5.0), 'max_depth': np.int64(4), 'learning_rate': np.float64(0.20333333333333334), 'gamma': np.float64(0.6666666666666666)}
Best F1 score from RandomizedSearchCV: 0.7282916506333468
Classification report for RandomizedSearchCV-tuned model:
              precision    recall  f1-score   support

           0       0.93      0.88      0.90      7431
           1       0.67      0.80      0.73      2338

    accuracy                           0.86      9769
   macro avg       0.80      0.84      0.82      9769
weighted avg       0.87      0.86      0.86      9769



### Tuning with Optuna

Using the code cell below as a starting point, tune your preferred model from above. You should tune the same 4-5 parameters as above using cross validation. Train a final model using the best hyperparameters and report your model performance.

In [50]:
# tuning with Optuna (tune more parameters than shown here)
def objective(trial):
    scale_pos_weight = trial.suggest_float('scale_pos_weight', 1.0, 10.0)
    max_depth = trial.suggest_int('max_depth', 3, 10)
    learning_rate = trial.suggest_float('learning_rate', 0.01, 0.3)
    gamma = trial.suggest_float('gamma', .5, 1)
    min_child_weight = trial.suggest_float('min_child_weight', 1, 5)
    
    # replace this placeholder model with your preferred model from above
    
    xgb_optuna = XGBClassifier(random_state=42, scale_pos_weight=scale_pos_weight, 
                               max_depth=max_depth,  learning_rate=learning_rate, gamma = gamma, min_child_weight = min_child_weight, enable_categorical=True)
    
    cv_scores = cross_val_score(xgb_optuna, X_train, y_train, cv=skf, scoring='f1')
    return cv_scores.mean()

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=20, show_progress_bar=True)

print(f'Best parameters from Optuna: {study.best_params}')
print(f'Best F1 score from Optuna: {study.best_value}')

# build preferred model from above with best parameters from Optuna and evaluate on the test set
xgb_optuna_best = XGBClassifier(random_state=42, scale_pos_weight=study.best_params['scale_pos_weight'], 
                                  max_depth=study.best_params['max_depth'], 
                                  learning_rate=study.best_params['learning_rate'], 
                                  gamma=study.best_params['gamma'],
                                  min_child_weight=study.best_params['min_child_weight'],
                                  enable_categorical=True)
xgb_optuna_best.fit(X_train, y_train)
y_pred_optuna = xgb_optuna_best.predict(X_test)
print(f'Classification report for Optuna-tuned model:\n{classification_report(y_test, y_pred_optuna)}')


[I 2026-04-15 16:03:47,644] A new study created in memory with name: no-name-5d5b9327-3fc9-4c92-b70d-6dfdc2876923


  0%|          | 0/20 [00:00<?, ?it/s]

[I 2026-04-15 16:03:48,319] Trial 0 finished with value: 0.6848421517748895 and parameters: {'scale_pos_weight': 9.223157981545718, 'max_depth': 10, 'learning_rate': 0.28431193532667326, 'gamma': 0.9270812470354138, 'min_child_weight': 2.3790940846663653}. Best is trial 0 with value: 0.6848421517748895.
[I 2026-04-15 16:03:48,935] Trial 1 finished with value: 0.7017973604383991 and parameters: {'scale_pos_weight': 4.5034764519061765, 'max_depth': 9, 'learning_rate': 0.24217848120505797, 'gamma': 0.6399132881746966, 'min_child_weight': 4.7781796107039725}. Best is trial 1 with value: 0.7017973604383991.
[I 2026-04-15 16:03:49,747] Trial 2 finished with value: 0.7165968220073097 and parameters: {'scale_pos_weight': 2.850784380374134, 'max_depth': 8, 'learning_rate': 0.070148873902931, 'gamma': 0.6321388293159045, 'min_child_weight': 4.276349253000814}. Best is trial 2 with value: 0.7165968220073097.
[I 2026-04-15 16:03:50,201] Trial 3 finished with value: 0.6495336877577206 and parameter

### Tuning results

In the markdown cell below describe your experience tuning with the different methods. Which produced the best results? Which do you prefer?


In my experience with tuning hyperparameter values in an XGBoost machine learning model, I found that it was much more effective and efficient tuning with Optuna. With tuning using grid search cross validation, I found that defining exact numbers for the model to iteratively run with all other combinations for the knobs was tedious and disappointing. Conversely, using optuna with the hyperparameter features that I found XGBoost performed well with, I was able to input a range of values, and it was able to algorithmically give me the similar performance with a lot less effort.